# Example queries: `restrict_avoid` (comstock_oedi)

Auto-generated from `tests/query_snapshots/restrict_avoid.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `restrict_single_state`

Annual electricity restricted to CO. Three equivalent restrict shapes (single-element list, scalar string, scalar wrapped in tuple) must all compile to the same SQL — covers the arg-normalization code path in _get_restrict_clauses.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (14, 4)
comstock_building_type  sample_count  units_count  electricity.total.energy_consumption..kwh
 FullServiceRestaurant         13461  4556.431396                               1.209202e+09
              Hospital            73   152.003100                               1.369473e+09
            LargeHotel          2815  3317.336942                               1.128185e+09
           LargeOffice          2271   849.927533                               1.546956e+09
          MediumOffice          8708  2418.404703                               1.298819e+09


## `restrict_two_states`

Annual electricity restricted to CO + WY.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['state'],
    restrict=[('state', ['CO', 'WY'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (2, 4)
state  sample_count  units_count  electricity.total.energy_consumption..kwh
   CO        134649 84415.712040                               1.600424e+10
   WY         18630 11666.235991                               1.768901e+09


## `restrict_state_plus_vintage`

CO + specific vintage bucket, electricity.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['vintage'],
    restrict=[('state', ['CO']), ('vintage', ['1980 to 1989'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (1, 4)
     vintage  sample_count  units_count  electricity.total.energy_consumption..kwh
1980 to 1989         21085 11994.992091                               2.733411e+09


## `avoid_building_type`

CO only, avoid one building type per schema.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    avoid=[('comstock_building_type', ['Warehouse'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (13, 4)
comstock_building_type  sample_count  units_count  electricity.total.energy_consumption..kwh
 FullServiceRestaurant         13461  4556.431396                               1.209202e+09
              Hospital            73   152.003100                               1.369473e+09
            LargeHotel          2815  3317.336942                               1.128185e+09
           LargeOffice          2271   849.927533                               1.546956e+09
          MediumOffice          8708  2418.404703                               1.298819e+09


## `avoid_building_type_multi`

CO only, avoid multiple building types via multi-element avoid list — exercises the NOT IN code path in _add_avoid (single-value avoid emits != instead).


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    avoid=[('comstock_building_type', ['Warehouse', 'SmallOffice'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (12, 4)
comstock_building_type  sample_count  units_count  electricity.total.energy_consumption..kwh
 FullServiceRestaurant         13461  4556.431396                               1.209202e+09
              Hospital            73   152.003100                               1.369473e+09
            LargeHotel          2815  3317.336942                               1.128185e+09
           LargeOffice          2271   849.927533                               1.546956e+09
          MediumOffice          8708  2418.404703                               1.298819e+09
